# 02 · Score forecasts and inspect disconnections

Use the same run ID as [01 · Fit and forecast](01_fit_and_forecast.ipynb).
Compare held-out multiclass Brier scores for the selector, pooled and local
fits, decoded persistence, and training prevalence. Lower scores are better.

Leave `DISCONNECT_ROIS` empty for forecast evaluation. To inspect a paired
model-based disconnection, provide a tuple of original recording-local ROI IDs.
For ROI-specific disconnections, use a run for the worm of interest.
The reported shift describes motif probabilities under the fitted model.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select the project Python 3.14 .venv kernel.")

working_directory = Path.cwd().resolve()
try:
    REPO = next(
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "effectome").is_dir()
    )
except StopIteration as error:
    raise RuntimeError("Open this notebook from inside the effectome application repository.") from error

project_src = str(REPO / "src")
if project_src not in sys.path:
    sys.path.insert(0, project_src)

print("Notebook kernel", sys.executable)


In [ ]:
RUN_ID = "motif-rollout-01"
OUTPUT_ROOT = REPO / "outputs" / "celegans-rollout" / RUN_ID
print("Run directory", OUTPUT_ROOT)

DISCONNECT_ROIS = ()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from effectome.workflows.celegans_rollout import summarize

evaluations = summarize(REPO, OUTPUT_ROOT, disconnect_rois=DISCONNECT_ROIS)
brier = pd.DataFrame(
    {entry["recording_id"]: entry["brier"] for entry in evaluations}
).T
display(brier.rename_axis("Recording"))
axes = brier.plot.bar(figsize=(10, 4), rot=15)
axes.set(xlabel="Recording", ylabel="Held-out Brier score (lower is better)")
axes.legend(title="Forecast", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
print("Saved", OUTPUT_ROOT / "summary.json")
for evaluation in evaluations:
    display({key: value for key, value in evaluation.items() if key != "brier"})
